In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional
from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI # Swap with ChatOllama or ChatGoogleGenerativeAI if needed
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_google_genai import ChatGoogleGenerativeAI


class Medication(BaseModel):
    name: str = Field(description="Name of the medication.")
    dosage: str = Field(description="Dosage and route, e.g., '1g IV' or '50mg PO'. If not found, output 'Not Documented'.")
    frequency: str = Field(description="Frequency, e.g., 'Daily' or 'BID'. If not found, output 'Not Documented'.")

class ExtractedClinicalData(BaseModel):
    admission_medications: List[Medication] = Field(description="Medications the patient was on upon admission or in the ER.")
    discharge_medications: List[Medication] = Field(description="Medications prescribed at discharge.")
    resolved_diagnoses: List[str] = Field(description="List of all diagnoses confirmed during the hospital stay.")
    pending_or_missing_fields: List[str] = Field(description="List any critical labs (like cultures) marked as 'pending', or missing data.")


from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional

class SourceDocument(BaseModel):
    doc_id: str
    doc_type: str
    timestamp: str
    raw_content: str

class SourceMappingTable(BaseModel):
    sentence_index: int
    summary_sentence: str
    verified_source_ids: List[str] = Field(default_factory=list)

class DischargeSummaryState(BaseModel):
    patient_id: str
    preprocessed_age: int
    preprocessed_gender: str
    chronological_docs: List[SourceDocument] = Field(default_factory=list)
    
    admission_medications: List[Dict[str, Any]] = Field(default_factory=list)
    discharge_medications: List[Dict[str, Any]] = Field(default_factory=list)
    resolved_diagnoses: List[str] = Field(default_factory=list)
    pending_or_missing_fields: List[str] = Field(default_factory=list)
    
    # --- The fields we were missing! ---
    self_eval_iteration: int = Field(default=0)
    max_eval_cycles: int = Field(default=3)
    is_summary_complete: bool = Field(default=False)
    
    reconciliation_escalation_flags: List[Dict[str, Any]] = Field(default_factory=list)
    source_attribution_ledger: List[SourceMappingTable] = Field(default_factory=list)
    
    current_draft: str = Field(default="")
    final_silver_summary: Optional[Dict[str, Any]] = None
    
    doctor_edited_gold_summary: str = Field(default="")
    calculated_edit_distance_reward: float = Field(default=0.0)
    step_execution_trace: List[str] = Field(default_factory=list)

# 3. THE EXTRACTOR NODE FUNCTION

def extraction_node(state: DischargeSummaryState) -> dict:
    """
    LangGraph Node: Reads chronological documents and extracts atomic entities.
    Returns a dictionary of state updates.
    """
    print("--- [NODE: EXTRACTION] ---")
    
    # Initialize the LLM (Using GPT-4o-mini or your local equivalent for fast extraction)
    # llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash") # If using Gemini
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0) 
    
    # Enforce the Pydantic schema
    structured_llm = llm.with_structured_output(ExtractedClinicalData)
    
    # Prepare the context from Phase 1
    context_blocks = []
    for doc in state.chronological_docs:
        context_blocks.append(f"[{doc.doc_type} - {doc.timestamp}]\n{doc.raw_content}")
    
    full_context = "\n\n".join(context_blocks)
    
    # The strict clinical prompt
    system_prompt = """
    You are an expert Clinical Data Extractor. Your job is to extract specific structured data from the provided clinical notes.
    
    CRITICAL GUARDRAILS:
    1. DO NOT HALLUCINATE. If a piece of information is not explicitly written in the text, you MUST output 'Not Documented'.
    2. If a lab test or culture is sent but the result is not yet available, add it to the `pending_or_missing_fields` list.
    3. Carefully distinguish between medications given IN THE ER/ADMISSION vs. medications prescribed AT DISCHARGE.
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Here are the patient's chronological documents:\n\n{context}\n\nExtract the required clinical data.")
    ])
    
    # Create the chain and execute
    extractor_chain = prompt | structured_llm
    
    try:
        print("Invoking LLM for structured extraction...")
        result: ExtractedClinicalData = extractor_chain.invoke({"context": full_context})
        
        trace_msg = f"Extraction successful. Found {len(result.resolved_diagnoses)} diagnoses, {len(result.admission_medications)} admit meds, {len(result.discharge_medications)} discharge meds."
        
        # Return the exact fields to update in the LangGraph State
        return {
            "admission_medications": [med.model_dump() for med in result.admission_medications],
            "discharge_medications": [med.model_dump() for med in result.discharge_medications],
            "resolved_diagnoses": result.resolved_diagnoses,
            "pending_or_missing_fields": result.pending_or_missing_fields,
            "step_execution_trace": [trace_msg] # Append to trace log
        }
        
    except Exception as e:
        error_msg = f"Extraction Node API Failure: {str(e)}"
        print(error_msg)
        return {
            "step_execution_trace": [error_msg]
        }



# 1. Load the text you saved from Phase 0
with open("patient_2_raw_transcript.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

import sys
import os

sys.path.append(os.path.abspath('..'))
# 2. Run it through the Phase 1 Parser we built earlier
from backend.services.document_parser import ClinicalDocumentParser # (Or just run your parser instance)
parser = ClinicalDocumentParser()
parsed_data = parser.parse_raw_text(patient_id="PATIENT_002", raw_text=raw_text)

# 3. Initialize the State
current_state = DischargeSummaryState(
    patient_id=parsed_data["patient_id"],
    preprocessed_age=parsed_data["preprocessed_age"],
    preprocessed_gender=parsed_data["preprocessed_gender"],
    chronological_docs=[SourceDocument(**doc) for doc in parsed_data["chronological_docs"]]
)

# 4. Run the Phase 2 Extraction Node!
state_updates = extraction_node(current_state)

import json
print("\n--- NODE OUTPUT ---")
print(json.dumps(state_updates, indent=2))

--- [NODE: EXTRACTION] ---
Invoking LLM for structured extraction...

--- NODE OUTPUT ---
{
  "admission_medications": [
    {
      "name": "INF NS",
      "dosage": "2 Bolus",
      "frequency": "Not Documented"
    },
    {
      "name": "INJ PAN",
      "dosage": "40mg/IV",
      "frequency": "Not Documented"
    },
    {
      "name": "INJ EMESET",
      "dosage": "4mg/IV",
      "frequency": "Not Documented"
    },
    {
      "name": "IV antibiotics",
      "dosage": "Not Documented",
      "frequency": "Not Documented"
    }
  ],
  "discharge_medications": [
    {
      "name": "TAB. RACIPER",
      "dosage": "40MG",
      "frequency": "Daily"
    },
    {
      "name": "TAB. EMESET",
      "dosage": "4MG",
      "frequency": "TID"
    },
    {
      "name": "TAB. OFLOX TZ",
      "dosage": "Not Documented",
      "frequency": "BID"
    },
    {
      "name": "TAB M STRONG",
      "dosage": "Not Documented",
      "frequency": "Daily"
    },
    {
      "name": "TAB. ZEDOTT",
 

In [1]:
print("hello")

hello


In [8]:
# ==========================================
# 1. DEFINE THE OUTPUT SCHEMA (GUARDRAILS)
# ==========================================
class ReconciliationFlag(BaseModel):
    medication_name: str = Field(description="The name of the medication involved in the discrepancy.")
    issue_type: str = Field(description="Must be 'Added', 'Stopped', or 'Changed'.")
    description: str = Field(description="Clear explanation of why this change lacks a documented clinical reason.")

class ReconciliationAndDraft(BaseModel):
    reconciliation_flags: List[ReconciliationFlag] = Field(
        description="List of flagged medication discrepancies. Leave empty if all changes have documented reasons."
    )
    silver_draft: str = Field(
        description="The drafted discharge summary formatted in clean Markdown."
    )

# ==========================================
# 2. THE RECONCILIATION NODE FUNCTION
# ==========================================
def reconciliation_node(state: DischargeSummaryState) -> dict:
    """
    LangGraph Node: Drafts the initial summary and audits medication changes.
    """
    print("--- [NODE: RECONCILIATION & GENERATION] ---")
    
    # Initialize the LLM (Using a stronger reasoning model here is recommended, e.g., GPT-4o)
    # llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash") # If using Gemini
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0) 
    structured_llm = llm.with_structured_output(ReconciliationAndDraft)
    
    # Re-compile context
    context_blocks = [f"[{doc.doc_type} - {doc.timestamp}]\n{doc.raw_content}" for doc in state.chronological_docs]
    full_context = "\n\n".join(context_blocks)
    
    system_prompt = """
    You are an expert Clinical Reconciliation Agent. Your job is to draft a Discharge Summary and audit medication safety.
    
    TASK 1: MEDICATION RECONCILIATION
    Compare the provided Admission Medications against the Discharge Medications. 
    If a medication was added, stopped, or changed, search the chronological clinical notes for a reason.
    If NO clear clinical reason is documented, you MUST flag it in the `reconciliation_flags`. Do not invent a reason.
    
    TASK 2: DRAFT THE SUMMARY
    Write a structured discharge summary using the following Markdown sections:
    ## 1. Patient Demographics & Admission/Discharge Dates
    ## 2. Diagnoses (Principal and Secondary)
    ## 3. Hospital Course
    ## 4. Discharge Medications (Clearly note changes from admission)
    ## 5. Pending Results
    ## 6. Follow-up Instructions
    ## 7. Discharge Condition
    
    CRITICAL RULE: If a required field cannot be sourced, explicitly mark it as "Not Documented / Pending Review".
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", """
        Patient Age: {age}
        Patient Gender: {gender}
        
        Admission Medications: {admit_meds}
        Discharge Medications: {discharge_meds}
        Resolved Diagnoses: {diagnoses}
        Pending/Missing Fields: {pending}
        
        Chronological Clinical Notes:
        {context}
        """)
    ])
    
    recon_chain = prompt | structured_llm
    
    try:
        print("Invoking LLM for drafting and reconciliation...")
        result: ReconciliationAndDraft = recon_chain.invoke({
            "age": state.preprocessed_age,
            "gender": state.preprocessed_gender,
            "admit_meds": state.admission_medications,
            "discharge_meds": state.discharge_medications,
            "diagnoses": state.resolved_diagnoses,
            "pending": state.pending_or_missing_fields,
            "context": full_context
        })
        
        flag_count = len(result.reconciliation_flags)
        trace_msg = f"Drafting complete. Identified {flag_count} unreasoned medication changes requiring escalation."
        
        return {
            "current_draft": result.silver_draft,
            "reconciliation_escalation_flags": [flag.model_dump() for flag in result.reconciliation_flags],
            "step_execution_trace": state.step_execution_trace + [trace_msg]
        }
        
    except Exception as e:
        error_msg = f"Reconciliation Node API Failure: {str(e)}"
        print(error_msg)
        return {
            "step_execution_trace": state.step_execution_trace + [error_msg]
        }
    

# 1. Update your existing state with the extraction results you just got
for key, value in state_updates.items():
    setattr(current_state, key, value)

# 2. Run the Reconciliation Node
recon_updates = reconciliation_node(current_state)

# 3. View the Results!
import json
print(f"\n--- FLAGS DETECTED: {len(recon_updates.get('reconciliation_escalation_flags', []))} ---")
print(json.dumps(recon_updates.get("reconciliation_escalation_flags", []), indent=2))

print("\n--- SILVER DRAFT ---")
print(recon_updates.get("current_draft", "No draft generated."))

--- [NODE: RECONCILIATION & GENERATION] ---
Invoking LLM for drafting and reconciliation...

--- FLAGS DETECTED: 2 ---
[
  {
    "medication_name": "TAB M STRONG",
    "issue_type": "Added",
    "description": "No clear clinical reason documented for adding this medication."
  },
  {
    "medication_name": "TAB. ENTRC",
    "issue_type": "Added",
    "description": "No clear clinical reason documented for adding this medication."
  }
]

--- SILVER DRAFT ---
## 1. Patient Demographics & Admission/Discharge Dates

*   **Patient Age:** Not Documented / Pending Review
*   **Patient Gender:** Unknown
*   **Admission Date:** Not Documented / Pending Review
*   **Discharge Date:** Not Documented / Pending Review

## 2. Diagnoses (Principal and Secondary)

*   **Principal Diagnosis:** Acute Gastroenteritis with Dehydration
*   **Secondary Diagnoses:**
    *   Urinary Tract Infection
    *   Grade-I fatty liver changes
    *   Hyponatremia (resolved)
    *   Acute Kidney Injury (resolved)
    *

In [9]:
# ==========================================
# 1. DEFINE THE OUTPUT SCHEMA (GUARDRAILS)
# ==========================================
class System2Evaluation(BaseModel):
    found_omissions: bool = Field(
        description="True if critical clinical facts (e.g., specific lab values, pain scales, treatments) from the source notes are missing in the draft."
    )
    omission_details: List[str] = Field(
        description="List of specific facts that were omitted. Leave empty if none."
    )
    revised_draft: str = Field(
        description="The updated discharge summary incorporating the missing facts. If no omissions were found, output the original draft exactly."
    )

# ==========================================
# 2. THE SELF-EVALUATION NODE FUNCTION
# ==========================================
def self_evaluation_node(state: DischargeSummaryState) -> dict:
    """
    LangGraph Node: Audits the current draft against raw sources to catch omissions.
    Limits to a hard cap of 3 cycles.
    """
    iteration = state.self_eval_iteration + 1
    print(f"--- [NODE: SYSTEM 2 EVALUATION] (Cycle {iteration}/{state.max_eval_cycles}) ---")
    
    # Check hard cap
    if iteration > state.max_eval_cycles:
        print("Maximum evaluation cycles reached. Finalizing draft.")
        return {
            "is_summary_complete": True,
            "step_execution_trace": state.step_execution_trace + ["Max System 2 cycles reached. Auto-finalizing."]
        }
    
    # Initialize the LLM
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0) 
    structured_llm = llm.with_structured_output(System2Evaluation)
    
    # Re-compile context
    context_blocks = [f"[{doc.doc_type} - {doc.timestamp}]\n{doc.raw_content}" for doc in state.chronological_docs]
    full_context = "\n\n".join(context_blocks)
    
    system_prompt = """
    You are an expert Clinical Auditor (System 2). 
    Your job is to read a drafted Discharge Summary and compare it against the raw chronological clinical notes.
    
    Look for OMISSIONS. Did the draft miss:
    - Specific critical lab values (e.g., exact Sodium, Creatinine, or WBC numbers)?
    - Specific imaging findings?
    - Pain scores or vital signs that indicate clinical trajectory?
    
    If you find omissions:
    1. Set 'found_omissions' to True.
    2. List the specific missing facts in 'omission_details'.
    3. Rewrite the draft in 'revised_draft' to seamlessly weave in these missing facts.
    
    If the draft captures all critical data accurately, set 'found_omissions' to False and return the original draft.
    DO NOT hallucinate data. Only use facts present in the raw clinical notes.
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", """
        CURRENT DRAFT:
        {draft}
        
        RAW CLINICAL NOTES:
        {context}
        """)
    ])
    
    eval_chain = prompt | structured_llm
    
    try:
        print("Invoking LLM for clinical auditing...")
        result: System2Evaluation = eval_chain.invoke({
            "draft": state.current_draft,
            "context": full_context
        })
        
        if result.found_omissions:
            trace_msg = f"System 2 Audit Cycle {iteration}: Found omissions: {', '.join(result.omission_details)}. Draft revised."
            is_complete = False
        else:
            trace_msg = f"System 2 Audit Cycle {iteration}: No omissions found. Draft is complete."
            is_complete = True
            
        return {
            "current_draft": result.revised_draft,
            "self_eval_iteration": iteration,
            "is_summary_complete": is_complete,
            "step_execution_trace": state.step_execution_trace + [trace_msg]
        }
        
    except Exception as e:
        error_msg = f"System 2 Node API Failure: {str(e)}"
        print(error_msg)
        return {
            "is_summary_complete": True, # Force complete on error to prevent infinite loops
            "step_execution_trace": state.step_execution_trace + [error_msg]
        }
    


# 1. Update the state with the Phase 3 draft
for key, value in recon_updates.items():
    setattr(current_state, key, value)

# 2. Run the System 2 Auditor
audit_updates = self_evaluation_node(current_state)

# 3. View the Results!
import json
print(f"\n--- AUDIT COMPLETE? {audit_updates.get('is_summary_complete')} ---")
print(f"--- TRACE LOG ---")
print(json.dumps(audit_updates.get("step_execution_trace", [])[-1], indent=2))

if not audit_updates.get('is_summary_complete'):
    print("\n--- REVISED DRAFT ---")
    print(audit_updates.get("current_draft"))

--- [NODE: SYSTEM 2 EVALUATION] (Cycle 1/3) ---
Invoking LLM for clinical auditing...

--- AUDIT COMPLETE? False ---
--- TRACE LOG ---
"System 2 Audit Cycle 1: Found omissions: Admission Date: 01.03.2020, Initial ER vital signs: Pulse 116 b/m, Respiratory Rate 22 b/m, Blood Pressure 87/50 mmHg, Temperature 98\u00b0F, SaO2 on air 96%., Initial ER Blood Glucose: 443 mg/dl., Initial ER Diagnosis: DKA., Initial Pain Score: 4/10., Specific urine routine findings: 10-12/hpf pus cells, 15-20/hpf epithelial cells., Specific stool routine findings: 2-3/hpf red blood cells.. Draft revised."

--- REVISED DRAFT ---
## 1. Patient Demographics & Admission/Discharge Dates

*   **Patient Age:** Not Documented / Pending Review
*   **Patient Gender:** Unknown
*   **Admission Date:** 01.03.2020
*   **Discharge Date:** Not Documented / Pending Review

## 2. Diagnoses (Principal and Secondary)

*   **Principal Diagnosis:** Acute Gastroenteritis with Dehydration
*   **Secondary Diagnoses:**
    *   Urinary 

In [10]:
# ==========================================
# 1. DEFINE THE OUTPUT SCHEMA (GUARDRAILS)
# ==========================================
class AttributionLedger(BaseModel):
    ledger: List[SourceMappingTable] = Field(
        description="A sequential list mapping every sentence in the summary to its source documents."
    )

# ==========================================
# 2. THE ATTRIBUTION NODE FUNCTION
# ==========================================
def attribution_node(state: DischargeSummaryState) -> dict:
    """
    LangGraph Node: Maps every sentence in the final draft back to the source document IDs.
    """
    print("--- [NODE: SOURCE ATTRIBUTION] ---")
    
    # Initialize the LLM (GPT-4o or equivalent)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
    structured_llm = llm.with_structured_output(AttributionLedger)
    
    # Re-compile context, but this time prominently feature the DOC_ID
    context_blocks = [f"[DOC_ID: {doc.doc_id} | TYPE: {doc.doc_type} | TIME: {doc.timestamp}]\n{doc.raw_content}" for doc in state.chronological_docs]
    full_context = "\n\n".join(context_blocks)
    
    system_prompt = """
    You are an expert Clinical Traceability Agent. 
    Your job is to read a final Discharge Summary and map every single sentence back to the raw source documents.
    
    INSTRUCTIONS:
    1. Break the CURRENT DRAFT down sentence by sentence.
    2. For each sentence, find the specific clinical notes that support the facts in that sentence.
    3. Return the exact sentence, its sequential index, and a list of the 'DOC_ID' strings for the supporting documents.
    4. If a sentence is just formatting (like a Markdown header) or general transition, leave the verified_source_ids list empty.
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", """
        CURRENT DRAFT:
        {draft}
        
        RAW CLINICAL NOTES WITH IDs:
        {context}
        """)
    ])
    
    attribution_chain = prompt | structured_llm
    
    try:
        print("Invoking LLM for sentence-level attribution mapping...")
        result: AttributionLedger = attribution_chain.invoke({
            "draft": state.current_draft,
            "context": full_context
        })
        
        trace_msg = f"Attribution complete. Mapped {len(result.ledger)} sentences to source documents."
        
        return {
            "source_attribution_ledger": [mapping.model_dump() for mapping in result.ledger],
            "step_execution_trace": state.step_execution_trace + [trace_msg]
        }
        
    except Exception as e:
        error_msg = f"Attribution Node API Failure: {str(e)}"
        print(error_msg)
        return {
            "step_execution_trace": state.step_execution_trace + [error_msg]
        }
    
# 1. Update the state with the Phase 4 revised draft
for key, value in audit_updates.items():
    setattr(current_state, key, value)

# 2. Run the Attribution Node
attribution_updates = attribution_node(current_state)

# 3. View the Results!
import json
print("\n--- ATTRIBUTION LEDGER (First 3 Sentences) ---")
ledger = attribution_updates.get("source_attribution_ledger", [])
print(json.dumps(ledger[:3], indent=2))

print("\n--- FINAL TRACE LOG ---")
for step in attribution_updates.get("step_execution_trace", []):
    print(f"- {step}")

--- [NODE: SOURCE ATTRIBUTION] ---
Invoking LLM for sentence-level attribution mapping...

--- ATTRIBUTION LEDGER (First 3 Sentences) ---
[
  {
    "sentence_index": 0,
    "summary_sentence": "## 1. Patient Demographics & Admission/Discharge Dates",
    "verified_source_ids": []
  },
  {
    "sentence_index": 1,
    "summary_sentence": "*   **Patient Age:** Not Documented / Pending Review",
    "verified_source_ids": []
  },
  {
    "sentence_index": 2,
    "summary_sentence": "*   **Patient Gender:** Unknown",
    "verified_source_ids": []
  }
]

--- FINAL TRACE LOG ---
- Extraction successful. Found 3 diagnoses, 4 admit meds, 8 discharge meds.
- Drafting complete. Identified 2 unreasoned medication changes requiring escalation.
- System 2 Audit Cycle 1: Found omissions: Admission Date: 01.03.2020, Initial ER vital signs: Pulse 116 b/m, Respiratory Rate 22 b/m, Blood Pressure 87/50 mmHg, Temperature 98°F, SaO2 on air 96%., Initial ER Blood Glucose: 443 mg/dl., Initial ER Diagnosis: D

--- INITIALIZING GRAPH ---
--- RUNNING FULL PIPELINE ---
--- [NODE: EXTRACTION] ---
Invoking LLM for structured extraction...
--- [NODE: RECONCILIATION & GENERATION] ---
Invoking LLM for drafting and reconciliation...
--- [NODE: SYSTEM 2 EVALUATION] (Cycle 1/3) ---
Invoking LLM for clinical auditing...
--- [NODE: SYSTEM 2 EVALUATION] (Cycle 2/3) ---
Invoking LLM for clinical auditing...
--- [NODE: SYSTEM 2 EVALUATION] (Cycle 3/3) ---
Invoking LLM for clinical auditing...
--- [NODE: SOURCE ATTRIBUTION] ---
Invoking LLM for sentence-level attribution mapping...

--- FINAL GRAPH TRACE ---
- Extraction successful. Found 2 diagnoses, 3 admit meds, 8 discharge meds.
- Drafting complete. Identified 2 unreasoned medication changes requiring escalation.
- System 2 Audit Cycle 1: Found omissions: Specific counts for pus cells (10-12/hpf) and epithelial cells (15-20/hpf) in urine routine., Specific count for red blood cells (2-3/hpf) in stool routine., Detailed imaging finding for ascending colon